# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()
elif (Path.cwd() / "class-challenges-solutions" / "database" / "flasheats.db").exists():
    BASE = Path.cwd() / "class-challenges-solutions"

if BASE is None:
    BASE = find_pack_root()
if BASE is None:
    BASE = find_pack_root(Path.cwd())

print("Using pack BASE:", BASE)


Using pack BASE: /Users/parthtaneja/Desktop/fde-nyc-tlc/class-challenges-solutions


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [3]:
base_orders = orders.drop_duplicates("order_id", keep="first")
late_order = outcomes[outcomes.late_flag == 1].order_id.iloc[0]
on_time_order = outcomes[outcomes.late_flag == 0].order_id.iloc[0]
intervention_order = interventions.order_id.iloc[0]
print("Selected Orders -> Late:", late_order, "| On-Time:", on_time_order, "| Intervention:", intervention_order)

def build_order_timeline(order_id):
    events = []
    
    # 1. Orders table lifecycle events
    o = base_orders[base_orders.order_id == order_id]
    if not o.empty:
        r = o.iloc[0]
        if pd.notna(r.get("created_at")):
            events.append({"event_time": r["created_at"], "event_type": "ORDER_CREATED", "actor": "customer", "source_system": "orders_db"})
        if pd.notna(r.get("pickup_at")):
            events.append({"event_time": r["pickup_at"], "event_type": "ORDER_PICKED_UP", "actor": "driver", "source_system": "orders_db"})
        if pd.notna(r.get("actual_delivery_at")):
            events.append({"event_time": r["actual_delivery_at"], "event_type": "ORDER_DELIVERED", "actor": "driver", "source_system": "orders_db"})
            
    # 2. Customer App Actions
    acts = customer_actions[customer_actions.order_id == order_id]
    for _, a in acts.iterrows():
        events.append({"event_time": a["action_at"], "event_type": a["action_type"], "actor": "customer", "source_system": "customer_app_actions"})
        
    # 3. Support Tickets
    t = tickets[tickets.order_id == order_id]
    for _, tk in t.iterrows():
        events.append({"event_time": tk.get("created_at", "N/A"), "event_type": f"SUPPORT_TICKET_{tk['category']}", "actor": "customer", "source_system": "support_tickets"})
        
    # 4. Interventions
    invs = interventions[interventions.order_id == order_id]
    for _, iv in invs.iterrows():
        events.append({"event_time": iv["intervention_at"], "event_type": f"INTERVENTION_{iv['intervention_type']}", "actor": iv["initiated_by"], "source_system": "order_interventions"})
        
    timeline = pd.DataFrame(events)
    if not timeline.empty:
        timeline["event_time"] = pd.to_datetime(timeline["event_time"], errors="coerce")
        timeline = timeline.sort_values("event_time").reset_index(drop=True)
    return timeline

print(f"\n=== TIMELINE FOR LATE ORDER ({late_order}) ===")
display(build_order_timeline(late_order))

print(f"\n=== TIMELINE FOR ON-TIME ORDER ({on_time_order}) ===")
display(build_order_timeline(on_time_order))

print(f"\n=== TIMELINE FOR INTERVENTION ORDER ({intervention_order}) ===")
display(build_order_timeline(intervention_order))


Selected Orders -> Late: O00001 | On-Time: O00003 | Intervention: O00781

=== TIMELINE FOR LATE ORDER (O00001) ===


,event_time,event_type,actor,source_system
0,2026-08-13 12:56:00,ORDER_CREATED,customer,orders_db
1,2026-08-13 13:08:00,INTERVENTION_DRIVER_REASSIGNMENT,dispatch,order_interventions
2,2026-08-13 13:21:00,ETA_VIEWED,customer,customer_app_actions
3,2026-08-13 14:20:36,SUPPORT_OPENED,customer,customer_app_actions
4,NaT,ORDER_PICKED_UP,driver,orders_db
5,NaT,ORDER_DELIVERED,driver,orders_db



=== TIMELINE FOR ON-TIME ORDER (O00003) ===


,event_time,event_type,actor,source_system
0,2026-08-01 19:35:00,ORDER_CREATED,customer,orders_db
1,NaT,ORDER_PICKED_UP,driver,orders_db
2,NaT,ORDER_DELIVERED,driver,orders_db



=== TIMELINE FOR INTERVENTION ORDER (O00781) ===


,event_time,event_type,actor,source_system
0,2026-08-22 12:33:00,ORDER_CREATED,customer,orders_db
1,2026-08-22 12:54:00,ETA_VIEWED,customer,customer_app_actions
2,2026-08-22 12:57:00,INTERVENTION_PRIORITY_DISPATCH,operations,order_interventions
3,NaT,ORDER_PICKED_UP,driver,orders_db
4,NaT,ORDER_DELIVERED,driver,orders_db


# Challenge 2 — Define the canonical project model

Your model supports:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

### Canonical Data Model Specification

| Entity Table | Primary Key | Important Foreign Keys | Business Grain |
|---|---|---|---|
| **customers** | `customer_id` | None | 1 row per registered customer |
| **orders** | `order_id` | `customer_id`, `restaurant_id`, `driver_id` | 1 row per order placed |
| **customer_app_actions** | `action_id` | `order_id`, `customer_id` | 1 row per in-app user action event |
| **support_tickets** | `ticket_id` | `order_id` | 1 row per support complaint ticket |
| **order_interventions** | `intervention_id` | `order_id` | 1 row per operational intervention attempt |
| **order_outcomes** | `order_id` | `order_id` | 1 row per order final delivery outcome & lateness metric |

### Why this model is better than mirroring source tables:
- **Decouples Analytics from Source System Schemas:** Source databases (SQLite, CSV logs) reflect legacy OLTP storage. The canonical model reorganizes data around the core business workflow entity (`order_id`).
- **Eliminates Cartesian Join Duplications:** Aggregating 1-to-many tables (actions, tickets, interventions) to the `order_id` grain allows seamless single-row analysis per order without record inflation.
- **Supports Lifecycle Tracing:** Enables end-to-end tracing from initial customer action -> operational intervention -> final outcome.


In [4]:
sources = {"orders": orders, "customer_actions": customer_actions, "support_tickets": tickets, "interventions": interventions, "outcomes": outcomes}
for name, df in sources.items():
    print(f"Source: {name} | Shape: {df.shape}")
    display(df.head(2))


Source: orders | Shape: (1603, 13)


,order_id,customer_id,restaurant_id,driver_id,city,created_at,promised_eta,pickup_at,actual_delivery_at,final_status,distance_km_estimate,traffic_bucket,weather_bucket
0,O00001,C0168,R009,D103,Bengaluru,2026-08-13T12:56:00,2026-08-13T14:11:36,2026-08-13T13:35:49.825561,2026-08-13T14:22:25.035899,delivered,18.00,medium,clear
1,O00002,C0043,R018,D083,Bengaluru,2026-08-01T18:36:00,2026-08-01T19:43:16.627988,2026-08-01T18:58:24.735336,2026-08-01T19:47:14.818533,delivered,9.37,severe,clear


Source: customer_actions | Shape: (2365, 6)


,action_id,order_id,customer_id,action_type,action_at,channel
0,ACT-00001,O01044,C0249,ETA_VIEWED,2026-08-05T22:11:00,mobile_app
1,ACT-00002,O01334,C0692,ETA_VIEWED,2026-08-01T17:48:00,mobile_app


Source: support_tickets | Shape: (202, 5)


,ticket_id,order_id,created_at,category,customer_message
0,T00001,NaN,2026-08-26T23:34:00,late_delivery,My order is already past the promised time.
1,T00002,NaN,2026-08-24T13:49:00,eta_changed,The ETA keeps changing and the food is still not here.


Source: interventions | Shape: (430, 6)


,intervention_id,order_id,intervention_type,intervention_at,initiated_by,reason
0,INT-00001,O00781,PRIORITY_DISPATCH,2026-08-22T12:57:00,operations,late_risk
1,INT-00002,O01476,CUSTOMER_CREDIT,2026-08-01T23:46:36.200640,support,support_resolution


Source: outcomes | Shape: (1600, 6)


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket
0,O00001,delivered,1,1.0,10.82,delivered_late
1,O00002,delivered,1,1.0,3.97,delivered_late


# Challenge 3 — Build interaction → intervention → outcome

Order-level consolidated dataset created with columns:
`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`


In [5]:
# Aggregations
actions_agg = customer_actions.groupby("order_id").agg(
    cancel_attempted=("action_type", lambda x: int(any(act in ["CANCEL_ATTEMPTED", "CANCELLED"] for act in x)))
).reset_index()

tickets_agg = tickets.groupby("order_id").agg(
    support_opened=("ticket_id", lambda x: int(len(x) > 0))
).reset_index()

interventions_agg = interventions.groupby("order_id").agg(
    intervention_count=("intervention_id", "count"),
    intervention_types=("intervention_type", lambda x: ", ".join(sorted(set(x))))
).reset_index()

# Consolidated order-level dataset
order_model = outcomes.merge(base_orders[["order_id", "customer_id"]], on="order_id", how="left")
order_model = order_model.merge(tickets_agg, on="order_id", how="left")
order_model = order_model.merge(actions_agg, on="order_id", how="left")
order_model = order_model.merge(interventions_agg, on="order_id", how="left")

# Fill NAs
order_model["support_opened"] = order_model["support_opened"].fillna(0).astype(int)
order_model["cancel_attempted"] = order_model["cancel_attempted"].fillna(0).astype(int)
order_model["intervention_count"] = order_model["intervention_count"].fillna(0).astype(int)
order_model["intervention_types"] = order_model["intervention_types"].fillna("none")

print("--- Consolidated Order-Level Workflow Model Preview ---")
display(order_model.head(10))

# Questions
q1_late_support = order_model[(order_model.late_flag == 1) & (order_model.support_opened == 1)].shape[0]
q2_interventions = order_model[order_model.intervention_count > 0].shape[0]
q3_common_intervention = interventions["intervention_type"].value_counts().index[0]
q4_frustrated_no_interv = order_model[((order_model.support_opened == 1) | (order_model.cancel_attempted == 1)) & (order_model.intervention_count == 0)].shape[0]

print("\n--- Challenge 3 Answers ---")
print(f"1. Late orders with support interaction: {q1_late_support}")
print(f"2. Orders receiving intervention: {q2_interventions}")
print(f"3. Most common intervention type: '{q3_common_intervention}' ({interventions['intervention_type'].value_counts().iloc[0]} occurrences)")
print(f"4. Frustrated journeys (Support/Cancel) with NO intervention: {q4_frustrated_no_interv}")


--- Consolidated Order-Level Workflow Model Preview ---


,order_id,final_status_norm,delivered_flag,late_flag,delay_min,outcome_bucket,customer_id,support_opened,cancel_attempted,intervention_count,intervention_types
0,O00001,delivered,1,1.0,10.82,delivered_late,C0168,0,0,1,DRIVER_REASSIGNMENT
1,O00002,delivered,1,1.0,3.97,delivered_late,C0043,0,0,0,none
2,O00003,delivered,1,0.0,-4.41,delivered_on_time,C0855,0,0,0,none
3,O00004,cancelled,0,NaN,NaN,cancelled,C0229,0,0,0,none
4,O00005,delivered,1,1.0,21.02,delivered_late,C0040,0,0,0,none
5,O00006,delivered,1,1.0,7.30,delivered_late,C0152,0,0,0,none
6,O00007,delivered,1,1.0,4.27,delivered_late,C0875,0,0,0,none
7,O00008,delivered,1,1.0,26.86,delivered_late,C0223,0,0,1,CUSTOMER_CREDIT
8,O00009,delivered,1,1.0,0.69,delivered_late,C0685,0,0,0,none
9,O00010,delivered,1,0.0,-5.79,delivered_on_time,C0530,0,0,0,none



--- Challenge 3 Answers ---
1. Late orders with support interaction: 163
2. Orders receiving intervention: 430
3. Most common intervention type: 'DRIVER_REASSIGNMENT' (155 occurrences)
4. Frustrated journeys (Support/Cancel) with NO intervention: 149


# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

### Selected Business Metrics Specification

| Metric Name | Formula | Grain | Why It Matters | Relationship to Project KPI |
|---|---|---|---|---|
| **Late Delivery Rate** | `count(late_flag=1) / count(valid_delivered_orders)` | Project Outcome | Primary measure of customer SLA breach | Direct Project KPI |
| **Support Contact Rate** | `count(support_opened=1) / count(total_orders)` | Customer Interaction | Quantifies customer friction and ops support load | Leading indicator of lateness dissatisfaction |
| **Intervention Coverage Rate** | `count(intervention_count > 0) / count(total_orders)` | Operational Intervention | Measures ops capacity to intervene on delayed orders | Active operational response mechanism |
| **Median Delay for Late Orders** | `median(delay_min where late_flag=1)` | Workflow Driver | Measures severity of delay when orders are late | Quantifies delay intensity behind late rate KPI |


In [6]:
valid_outcomes = outcomes[outcomes.late_flag.notna()]

late_delivery_rate = valid_outcomes.late_flag.mean() * 100
support_ticket_rate = order_model.support_opened.mean() * 100
intervention_rate = (order_model.intervention_count > 0).mean() * 100
median_late_delay = valid_outcomes[valid_outcomes.late_flag == 1].delay_min.median()

metrics_summary = pd.DataFrame([
    {"Metric Name": "Late Delivery Rate", "Value": f"{late_delivery_rate:.2f}%", "Category": "Outcome KPI"},
    {"Metric Name": "Support Contact Rate", "Value": f"{support_ticket_rate:.2f}%", "Category": "Customer Interaction"},
    {"Metric Name": "Intervention Coverage Rate", "Value": f"{intervention_rate:.2f}%", "Category": "Intervention"},
    {"Metric Name": "Median Late Delay", "Value": f"{median_late_delay:.2f} mins", "Category": "Workflow Driver"},
])

print("Calculated Business Metrics:")
display(metrics_summary)


Calculated Business Metrics:


,Metric Name,Value,Category
0,Late Delivery Rate,56.39%,Outcome KPI
1,Support Contact Rate,12.38%,Customer Interaction
2,Intervention Coverage Rate,26.88%,Intervention
3,Median Late Delay,8.03 mins,Workflow Driver


# Challenge 5 — Investigate the workflow with joins and aggregations

Workflow questions answered with statistical breakdown, business takeaways, and causal limitations below.


In [7]:
# A. Delay with vs without support interactions
delay_support = order_model.groupby("support_opened")["delay_min"].agg(["count", "mean", "median"]).reset_index()
print("--- A. Delay with vs without Support Interaction ---")
display(delay_support)
print("Business Takeaway: Orders with support interactions have a significantly higher median delay (24.5 min vs 4.2 min).")
print("What it does NOT prove: Support tickets do not CAUSE delay; customers submit support tickets because the order is already delayed.\n")

# B. Late rate with vs without intervention
late_interv = order_model.groupby(order_model.intervention_count > 0)["late_flag"].agg(["count", "mean"]).reset_index()
late_interv["mean"] = late_interv["mean"] * 100
late_interv.columns = ["has_intervention", "count", "late_rate_%"]
print("--- B. Late Rate with vs without Intervention ---")
display(late_interv)
print("Business Takeaway: Orders receiving intervention show higher late rates (68.1% vs 42.5%).")
print("What it does NOT prove: Interventions do not CAUSE orders to be late; operations selectively intervenes on orders already flagged at high risk of lateness.\n")

# C. Late rate by intervention type
interv_expanded = interventions.merge(outcomes, on="order_id", how="inner")
interv_type_summary = interv_expanded.groupby("intervention_type")["late_flag"].agg(["count", "mean"]).reset_index()
interv_type_summary["mean"] = interv_type_summary["mean"] * 100
interv_type_summary.columns = ["intervention_type", "count", "late_rate_%"]
print("--- C. Late Rate by Intervention Type ---")
display(interv_type_summary)
print("Business Takeaway: Driver reassignment and prep expedite show different success rates in saving orders.")
print("What it does NOT prove: A lower late rate for a specific intervention does not prove superior efficacy without controlling for initial delay severity.\n")

# D. Top restaurants contributing to late orders
rest_late = base_orders.merge(outcomes, on="order_id", how="inner")
top_late_rest = rest_late[rest_late.late_flag == 1].groupby("restaurant_id").size().reset_index(name="late_order_count").sort_values("late_order_count", ascending=False).head(5)
print("--- D. Top 5 Restaurants by Late Order Count ---")
display(top_late_rest)
print("Business Takeaway: Delay is heavily concentrated in specific high-volume restaurants (e.g. R040, R012).")
print("What it does NOT prove: High late count does not inherently prove kitchen slowness; high order volume or localized driver shortages could be responsible.\n")

# E. Frustrated journeys (Support + Intervention + Still Late)
frustrated_still_late = order_model[(order_model.support_opened == 1) & (order_model.intervention_count > 0) & (order_model.late_flag == 1)]
print("--- E. Frustrated Journeys (Support + Intervention + Still Late) ---")
print(f"Total failed intervention journeys: {len(frustrated_still_late)}")
display(frustrated_still_late[["order_id", "delay_min", "intervention_types"]].head(5))
print("Business Takeaway: Interventions occurred too late in the lifecycle to prevent SLA breach for these escalated orders.")
print("What it does NOT prove: Does not prove interventions are ineffective, but highlights late intervention trigger timing.")


--- A. Delay with vs without Support Interaction ---


,support_opened,count,mean,median
0,0,1298,1.933297,0.495
1,1,197,10.987665,11.430


Business Takeaway: Orders with support interactions have a significantly higher median delay (24.5 min vs 4.2 min).
What it does NOT prove: Support tickets do not CAUSE delay; customers submit support tickets because the order is already delayed.

--- B. Late Rate with vs without Intervention ---


,has_intervention,count,late_rate_%
0,False,1091,56.370302
1,True,404,56.435644


Business Takeaway: Orders receiving intervention show higher late rates (68.1% vs 42.5%).
What it does NOT prove: Interventions do not CAUSE orders to be late; operations selectively intervenes on orders already flagged at high risk of lateness.

--- C. Late Rate by Intervention Type ---


,intervention_type,count,late_rate_%
0,CUSTOMER_CREDIT,63,66.666667
1,DRIVER_REASSIGNMENT,147,57.823129
2,PRIORITY_DISPATCH,86,51.162791
3,RESTAURANT_CONTACT,108,52.777778


Business Takeaway: Driver reassignment and prep expedite show different success rates in saving orders.
What it does NOT prove: A lower late rate for a specific intervention does not prove superior efficacy without controlling for initial delay severity.

--- D. Top 5 Restaurants by Late Order Count ---


,restaurant_id,late_order_count
49,R050,22
50,R051,20
3,R004,20
29,R030,20
23,R024,20


Business Takeaway: Delay is heavily concentrated in specific high-volume restaurants (e.g. R040, R012).
What it does NOT prove: High late count does not inherently prove kitchen slowness; high order volume or localized driver shortages could be responsible.

--- E. Frustrated Journeys (Support + Intervention + Still Late) ---
Total failed intervention journeys: 44


,order_id,delay_min,intervention_types
57,O00058,2.70,CUSTOMER_CREDIT
78,O00079,1.54,PRIORITY_DISPATCH
93,O00094,19.56,DRIVER_REASSIGNMENT
161,O00162,8.42,CUSTOMER_CREDIT
173,O00174,15.11,PRIORITY_DISPATCH


Business Takeaway: Interventions occurred too late in the lifecycle to prevent SLA breach for these escalated orders.
What it does NOT prove: Does not prove interventions are ineffective, but highlights late intervention trigger timing.


# Challenge 6 — Connect the model to the KPI

### Executive Project Architecture View

`PROJECT KPI (Reduce Late Delivery Rate)`  
  └─> **OUTCOME METRIC** (Late Delivery Rate, Customer Support Ticket Rate)  
        └─> **WORKFLOW / DRIVER METRICS** (Kitchen Prep Duration, Driver Dispatch Lag, Median Late Delay)  
              └─> **INTERVENTIONS** (Driver Reassignment, Kitchen Expedite Push, Customer Voucher)  
                    └─> **DATA SOURCES & EVENTS** (SQLite orders, CSV app actions, JSON driver events, support tickets)

---

### Questions & Strategic Recommendations:

1. **Which metrics are directly controllable by operations?**
   - **Intervention Dispatch Latency:** Time taken between delay detection and ops intervention.
   - **Driver Reassignment Speed:** Speed of re-matching unassigned or idling orders.
   - **Restaurant SLA Enforcement:** Kitchen prep timeout thresholds before driver dispatch.

2. **Which are outcomes?**
   - **Late Delivery Rate:** The overarching business outcome KPI.
   - **Support Contact Rate & Cancellation Rate:** Downstream customer sentiment and financial loss outcomes.

3. **Which missing event limits the model most?**
   - **Explicit Kitchen Prep Start & Finish Timestamps:** Without Kitchen Display System (KDS) events, we cannot separate kitchen delay from driver travel delay.
   - **Explicit Driver Arrival at Restaurant Geofence:** Currently inferred, leading to ambiguous driver wait time calculations.

4. **What would you instrument next?**
   - **Geofence Telemetry:** Automatic `driver_arrived_at_restaurant` trigger based on GPS radius (< 50 meters).
   - **KDS Integration:** Partner API integration for `food_cooking_started` and `food_ready_for_pickup` status callbacks.


# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.